# Parse `processInformation` from raw ecoSpold XML

`data/raw/ecoSpold files/` holds 11,947 raw BAFU-2026 process XML files in EcoSpold **v1** format (no namespace, unlike the processed EcoSpold2 `.spold` files under `data/processed/ecospold2-biosphere310/`). Each file's `dataset/metaInformation/processInformation` section carries the process's core metadata as XML attributes. This notebook parses that section, across a small sample first, into one wide pandas DataFrame: one row per process, columns prefixed by subsection.

In [1]:
import xml.etree.ElementTree as ET
from pathlib import Path

import pandas as pd

In [2]:
RAW_DIR = Path("../data/raw/ecoSpold files")

all_files = sorted(RAW_DIR.glob("process_*.xml"))
print(f"{len(all_files):,} raw ecoSpold files found")

# SAMPLE_SIZE = 50   # validated on a sample first; now parsing the full corpus
sample_files = all_files
print(f"parsing {len(sample_files):,} files")

11,947 raw ecoSpold files found
parsing 11,947 files


In [3]:
def parse_process_information(path: Path) -> dict:
    """Flatten a raw ecoSpold file's processInformation section into one row.

    Every subsection's attributes are prefixed with the subsection name
    (e.g. referenceFunction_name, geography_location). timePeriod additionally
    carries startDate/endDate as text children rather than attributes.
    """
    # filenames look like process_<uuid>.xml
    uuid = path.stem.removeprefix("process_")

    root = ET.parse(path).getroot()
    pi = root.find("dataset/metaInformation/processInformation")

    row: dict = {"uuid": uuid, "file": path.name}

    for section in ("referenceFunction", "geography", "technology", "dataSetInformation"):
        elem = pi.find(section)
        if elem is None:
            continue
        for key, value in elem.attrib.items():
            row[f"{section}_{key}"] = value

    time_period = pi.find("timePeriod")
    if time_period is not None:
        for key, value in time_period.attrib.items():
            row[f"timePeriod_{key}"] = value
        for child_tag in ("startDate", "endDate"):
            child = time_period.find(child_tag)
            row[f"timePeriod_{child_tag}"] = child.text if child is not None else None

    return row

In [4]:
rows = [parse_process_information(f) for f in sample_files]
df = pd.DataFrame(rows)

print("shape:", df.shape)
pd.set_option("display.max_columns", 40)
pd.set_option("display.max_colwidth", 40)
df.head()

shape: (11947, 33)


,uuid,file,referenceFunction_amount,referenceFunction_category,referenceFunction_datasetRelatesToProduct,referenceFunction_generalComment,referenceFunction_includedProcesses,referenceFunction_infrastructureProcess,referenceFunction_localCategory,referenceFunction_localName,referenceFunction_localSubCategory,referenceFunction_name,referenceFunction_subCategory,referenceFunction_unit,geography_location,geography_text,technology_text,dataSetInformation_energyValues,dataSetInformation_impactAssessmentResult,dataSetInformation_internalVersion,dataSetInformation_languageCode,dataSetInformation_localLanguageCode,dataSetInformation_timestamp,dataSetInformation_type,dataSetInformation_version,timePeriod_dataValidForEntirePeriod,timePeriod_text,timePeriod_startDate,timePeriod_endDate,referenceFunction_infrastructureIncluded,referenceFunction_CASNumber,referenceFunction_formula,referenceFunction_text
0,0004e814-c18d-42e2-a3f7-ce1fa51a3c2c,process_0004e814-c18d-42e2-a3f7-ce1f...,1.0,nuclear waste,true,For operation is meant the input of ...,This module includes: concrete and e...,false,nuclear waste,"Radioactive waste, in final reposito...",unspecified,"Radioactive waste, in final reposito...",unspecified,m3,CH,<null>,Currently available knowledge.,0,false,0.0,en,de,2004-04-02T00:00:00.000+02:00,1,2023,true,undefined,2000-01,2000-12,NaN,NaN,NaN,NaN
1,0016d799-566a-3a58-bf88-48e3a349ee00,process_0016d799-566a-3a58-bf88-48e3...,1.0,electricity,true,Electricity domestic net production ...,It includes the shares of domestic e...,false,electricity,"xxx Electricity, production mix MX",notMaintained,"xxx Electricity, production mix MX",notMaintained,kWh,MX,Data apply to public and self produc...,No technology description is provide...,0,false,0.0,en,de,2012-06-22T00:00:00.000+02:00,1,2023,true,Time period of statistics used (2008).,2008-01,2008-12,NaN,NaN,NaN,NaN
2,00173db7-7590-3ac7-bf27-a41684347176,process_00173db7-7590-3ac7-bf27-a416...,1.0,construction,true,Manufacturing process is considered ...,<null>,false,construction,"Acrylic varnish, 87.5% in H2O, at plant",paints,"Acrylic varnish, 87.5% in H2O, at plant",paints,kg,RER,The literature source used bases on ...,unknown,0,false,0.0,en,de,2003-07-25T00:00:00.000+02:00,1,2023,true,undefined,2000-01,2000-12,NaN,NaN,NaN,NaN
3,0017b758-86e7-499e-b582-8cd4f2a1256d,process_0017b758-86e7-499e-b582-8cd4...,1.0,waste management,true,na;\nUUID: 0017b758-86e7-499e-b582-8...,<null>,false,waste management,"Disposal, glazing, 3-IV, U=0.6 W/m2K...",building demolition,"Disposal, glazing, 3-IV, U=0.6 W/m2K...",building demolition,m2,CH,undefined,undefined,0,false,0.0,en,de,2020-12-07T00:00:00.000+01:00,1,2023,true,undefined,2020-01,2020-12,NaN,NaN,NaN,NaN
4,001835f5-ba6d-361a-8990-7c894d80c087,process_001835f5-ba6d-361a-8990-7c89...,1,natural gas,true,The process is normalized on the gas...,This dataset describes the transport...,false,natural gas,"Natural gas, liquefied, production A...",production,"Natural gas, liquefied, production A...",production,Nm3,KW,not known,The distances between liquefaction a...,0,false,1.0,en,de,2025-07-16T13:40:59,1,1,true,Transport modes investigated for 2023.,None,None,true,NaN,NaN,NaN


## Sanity checks

In [5]:
print("columns:", list(df.columns))
print()
print("uuid unique:", df["uuid"].is_unique)
print()
print("null counts per column:")
print(df.isna().sum().sort_values(ascending=False))

columns: ['uuid', 'file', 'referenceFunction_amount', 'referenceFunction_category', 'referenceFunction_datasetRelatesToProduct', 'referenceFunction_generalComment', 'referenceFunction_includedProcesses', 'referenceFunction_infrastructureProcess', 'referenceFunction_localCategory', 'referenceFunction_localName', 'referenceFunction_localSubCategory', 'referenceFunction_name', 'referenceFunction_subCategory', 'referenceFunction_unit', 'geography_location', 'geography_text', 'technology_text', 'dataSetInformation_energyValues', 'dataSetInformation_impactAssessmentResult', 'dataSetInformation_internalVersion', 'dataSetInformation_languageCode', 'dataSetInformation_localLanguageCode', 'dataSetInformation_timestamp', 'dataSetInformation_type', 'dataSetInformation_version', 'timePeriod_dataValidForEntirePeriod', 'timePeriod_text', 'timePeriod_startDate', 'timePeriod_endDate', 'referenceFunction_infrastructureIncluded', 'referenceFunction_CASNumber', 'referenceFunction_formula', 'referenceFunct

In [6]:
df["referenceFunction_amount"] = pd.to_numeric(df["referenceFunction_amount"])

cols = [
    "uuid",
    "referenceFunction_name",
    "referenceFunction_category",
    "referenceFunction_subCategory",
    "referenceFunction_unit",
    "geography_location",
    "timePeriod_startDate",
    "timePeriod_endDate",
]
df[cols].head(10)

,uuid,referenceFunction_name,referenceFunction_category,referenceFunction_subCategory,referenceFunction_unit,geography_location,timePeriod_startDate,timePeriod_endDate
0,0004e814-c18d-42e2-a3f7-ce1fa51a3c2c,"Radioactive waste, in final reposito...",nuclear waste,unspecified,m3,CH,2000-01,2000-12
1,0016d799-566a-3a58-bf88-48e3a349ee00,"xxx Electricity, production mix MX",electricity,notMaintained,kWh,MX,2008-01,2008-12
2,00173db7-7590-3ac7-bf27-a41684347176,"Acrylic varnish, 87.5% in H2O, at plant",construction,paints,kg,RER,2000-01,2000-12
3,0017b758-86e7-499e-b582-8cd4f2a1256d,"Disposal, glazing, 3-IV, U=0.6 W/m2K...",waste management,building demolition,m2,CH,2020-01,2020-12
4,001835f5-ba6d-361a-8990-7c894d80c087,"Natural gas, liquefied, production A...",natural gas,production,Nm3,KW,None,None
5,001b52f2-e932-45f9-a199-c82f584d8dab,"Glass cullets, recovered from CdTe P...",waste management,recycling,kg,DE,2014-01,2017-12
6,001f4c49-9ea8-3342-8a17-dfa8db07e634,"xxx Heat, at hot water tank, solar+e...","energy, obsolete","others, obsolete",MJ,CH,9999-01,9999-12
7,00240fd1-4fb6-3602-9d37-176288c85424,"Electrolyte, KOH, LiOH additive, at ...",electronics,devices\module\component,kg,GLO,2004-01,2004-12
8,002534dc-d30b-3a2e-aef7-1fb791a4b6fe,"xxx Electricity mix, medium voltage,...",electricity,notMaintained,kWh,SK,2018-01,2018-12
9,002b101b-940c-46e6-bc77-483d43f7d3eb,"Vibratory roller, 15t, self-propelle...",construction processes,building equipment\machinery,hr,CH,None,None


In [7]:
out_path = Path("processInformation.parquet")
df.to_parquet(out_path, index=False)
print(f"wrote {len(df):,} rows x {df.shape[1]} cols to {out_path.resolve()}")

wrote 11,947 rows x 33 cols to C:\Users\rusai1\PycharmProjects\trailrunner\database\processInformation.parquet


## Group by category, export as JSON

One array of objects, each object a single `{category: [processes...]}` pair; every process keeps all 33 columns.

In [8]:
import json


def clean_record(record: dict) -> dict:
    """Replace NaN/NaT with None so every value is valid JSON."""
    return {k: (None if pd.isna(v) else v) for k, v in record.items()}


by_category = []
for category, group in df.groupby("referenceFunction_category", dropna=False, sort=True):
    records = [clean_record(r) for r in group.to_dict(orient="records")]
    by_category.append({category: records})

json_path = Path("processInformation_by_category.json")
with json_path.open("w", encoding="utf-8") as f:
    json.dump(by_category, f, ensure_ascii=False, indent=2, allow_nan=False)

n_processes = sum(len(next(iter(d.values()))) for d in by_category)
print(f"wrote {len(by_category)} categories, {n_processes:,} processes to {json_path.resolve()}")
print(f"file size: {json_path.stat().st_size / 1e6:.1f} MB")

wrote 59 categories, 11,947 processes to C:\Users\rusai1\PycharmProjects\trailrunner\database\processInformation_by_category.json
file size: 30.3 MB


## Within each category, subset by cleaned process name

Same category grouping as above, but each category's value is now an object keyed by a cleaned `referenceFunction_name` (lowercased, spaces and special characters stripped) instead of a flat list. Two different raw names that clean to the same key are merged into one array. Rows whose `referenceFunction_subCategory` is `notMaintained` (case-insensitive) are excluded first.

In [9]:
import re


def slugify(name: str) -> str:
    """Lowercase, strip everything but letters and digits."""
    return re.sub(r"[^a-z0-9]", "", name.lower())


excluded_mask = df["referenceFunction_subCategory"].str.lower() == "notmaintained"
print(f"excluding {excluded_mask.sum():,} rows with subCategory 'notMaintained'")

filtered = df.loc[~excluded_mask].copy()
filtered["_name_slug"] = filtered["referenceFunction_name"].map(slugify)

by_category_by_name = []
for category, cat_group in filtered.groupby("referenceFunction_category", dropna=False, sort=True):
    by_name = {}
    for slug, slug_group in cat_group.groupby("_name_slug", sort=True):
        records = [
            clean_record({k: v for k, v in r.items() if k != "_name_slug"})
            for r in slug_group.to_dict(orient="records")
        ]
        by_name[slug] = records
    by_category_by_name.append({category: by_name})

n_processes = sum(len(procs) for d in by_category_by_name for procs in next(iter(d.values())).values())
n_slugs = sum(len(next(iter(d.values()))) for d in by_category_by_name)
print(f"{len(by_category_by_name)} categories, {n_slugs:,} name-subsets, {n_processes:,} processes")

by_name_json_path = Path("processInformation_by_category_by_name.json")
with by_name_json_path.open("w", encoding="utf-8") as f:
    json.dump(by_category_by_name, f, ensure_ascii=False, indent=2, allow_nan=False)

print(f"wrote to {by_name_json_path.resolve()}")
print(f"file size: {by_name_json_path.stat().st_size / 1e6:.1f} MB")

excluding 441 rows with subCategory 'notMaintained'


59 categories, 8,138 name-subsets, 11,506 processes


wrote to C:\Users\rusai1\PycharmProjects\trailrunner\database\processInformation_by_category_by_name.json
file size: 30.6 MB


## Colliding name-slugs

Within a category, two different raw `referenceFunction_name` values can clean to the same slug (their processes get merged into one array above). Export those as a CSV table: one row per colliding `(category, slug)` group with both original names side by side. Most collisions here are a stripped `<`/`>` (e.g. `<30kW` vs `>30kW`) &mdash; meaningfully different processes, not true duplicates, so worth a manual look.

In [10]:
name_grp = filtered.groupby(["referenceFunction_category", "_name_slug"])["referenceFunction_name"].unique()
colliding = name_grp[name_grp.map(len) > 1]
assert colliding.map(len).max() == 2, "expected only pairwise collisions"

collisions_df = pd.DataFrame(
    [
        {
            "referenceFunction_category": category,
            "name_slug": slug,
            "name_1": names[0],
            "name_2": names[1],
        }
        for (category, slug), names in colliding.items()
    ]
)

collisions_path = Path("colliding_name_slugs.csv")
collisions_df.to_csv(collisions_path, index=False)
print(f"wrote {len(collisions_df)} colliding name pairs to {collisions_path.resolve()}")
collisions_df

wrote 9 colliding name pairs to C:\Users\rusai1\PycharmProjects\trailrunner\database\colliding_name_slugs.csv


,referenceFunction_category,name_slug,name_1,name_2
0,compressed air,compressedairaveragegeneration30kw8b...,"Compressed air, average generation, ...","Compressed air, average generation, ..."
1,compressed air,compressedairaverageinstallation30kw...,"Compressed air, average installation...","Compressed air, average installation..."
2,compressed air,compressedairoptimisedgeneration30kw...,"Compressed air, optimised generation...","Compressed air, optimised generation..."
3,compressed air,xxcompressedairoptimisedinstallation...,"xx Compressed air, optimised install...","xx Compressed air, optimised install..."
4,computers & network,usecomputerdesktopwithcrtmonitoroffi...,"Use, computer, desktop, with CRT mon...","Use, computer, desktop with CRT moni..."
5,construction materials,solidwoodsprucefirlarchswitzerlandai...,"Solid wood, spruce, fir, larch Switz...","Solid wood, spruce / fir / larch Swi..."
6,electronics,capacitorelectrolytetype2cmheightatp...,"Capacitor, electrolyte type, > 2cm h...","Capacitor, electrolyte type, < 2cm h..."
7,"energy, obsolete",xxheatnaturalgasatboilermodulating100kw,"xx Heat, natural gas, at boiler modu...","xx Heat, natural gas, at boiler modu..."
8,heat,naturalgasburnedinboilermodulating100kw,"Natural gas, burned in boiler modula...","Natural gas, burned in boiler modula..."


## `fuels` / `natural gas`: exact-match name counts

Filter to `referenceFunction_category == "fuels"` and `referenceFunction_subCategory == "natural gas"`, group by the exact (unslugified) `referenceFunction_name`, and count.

In [11]:
natural_gas = df[
    (df["referenceFunction_category"] == "fuels")
    & (df["referenceFunction_subCategory"] == "natural gas")
]
print(f"{len(natural_gas)} processes in fuels / natural gas")

name_counts = (
    natural_gas.groupby("referenceFunction_name")
    .size()
    .reset_index(name="count")
    .sort_values("count", ascending=False)
    .reset_index(drop=True)
)
print(f"{len(name_counts)} distinct exact names")
name_counts

56 processes in fuels / natural gas
2 distinct exact names


,referenceFunction_name,count
0,"Transport, natural gas, onshore pipe...",42
1,"Transport, natural gas, offshore pip...",14


### Exchange count per process, "Transport, natural gas, offshore pipeline, long distance"

Re-opens each raw XML file for this exact-name group (still scoped to fuels / natural gas, 14 processes) and counts its `<exchange>` elements under `dataset/flowData`.

In [12]:
offshore_group = natural_gas[
    natural_gas["referenceFunction_name"] == "Transport, natural gas, offshore pipeline, long distance"
]
print(f"{len(offshore_group)} processes in this exact-name group")

exchange_counts = []
for _, row in offshore_group.iterrows():
    path = RAW_DIR / row["file"]
    root = ET.parse(path).getroot()
    flow_data = root.find("dataset/flowData")
    exchange_counts.append(
        {
            "uuid": row["uuid"],
            "geography_location": row["geography_location"],
            "n_exchanges": len(flow_data.findall("exchange")),
        }
    )

exchange_counts_df = pd.DataFrame(exchange_counts).sort_values("geography_location").reset_index(drop=True)
exchange_counts_df

14 processes in this exact-name group


,uuid,geography_location,n_exchanges
0,5c52cc7d-2c0c-3a6d-aaaa-9bcf709d422d,AZ,6
1,4653738d-e002-34ea-877f-d84c95323766,DZ,15
2,00ce87de-5e7f-3e8f-8fa0-ab2247a6537b,GB,15
3,42bbe9a2-7024-3474-bb26-23d2513e92a5,ID,15
4,cfca874d-0c75-3fee-bc36-b14c91d54988,IR,15
5,73cbf3da-a567-3228-8f9e-0c0fa6f52daa,IT,6
6,4d3be0ab-7b25-3586-ada0-ff1c81227044,LY,15
7,16cf47b5-6d04-354a-b88e-1af446dd3fc8,MY,15
8,6e65b625-e68f-352f-afed-f051505eb360,NL,15
9,032e8c77-8398-31bd-b400-d989f3a5cbc5,NO,15


### Common vs. non-common exchanges across those 14 locations

For each raw XML file, `<exchange>` elements carry `name` and `meanValue` as attributes (not child text). Pivot to exchange name x location, `meanValue` in each cell. Split into exchanges present at all 14 locations ("common") vs. present at only some ("non-common") &mdash; explains the 6-vs-15 exchange-count split seen above (AZ/IT/UA only have the common ones).

In [13]:
exchange_rows = []
for _, row in offshore_group.iterrows():
    root = ET.parse(RAW_DIR / row["file"]).getroot()
    flow_data = root.find("dataset/flowData")
    for ex in flow_data.findall("exchange"):
        exchange_rows.append(
            {
                "location": row["geography_location"],
                "exchange_name": ex.get("name"),
                "meanValue": ex.get("meanValue"),
            }
        )

exchange_long = pd.DataFrame(exchange_rows)
exchange_pivot = exchange_long.pivot_table(
    index="exchange_name", columns="location", values="meanValue", aggfunc="first"
)

locations = sorted(offshore_group["geography_location"])
present_count = exchange_pivot.notna().sum(axis=1)

common_exchanges = exchange_pivot[present_count == len(locations)].sort_index()
noncommon_exchanges = exchange_pivot[present_count < len(locations)].sort_index()

print(f"{len(common_exchanges)} common exchanges (present at all {len(locations)} locations)")
print(f"{len(noncommon_exchanges)} non-common exchanges (present at only some locations)")

# write both tables to one CSV, non-common block starting 3 blank rows below the common one
table_path = Path("offshore_pipeline_exchanges.csv")
with table_path.open("w", encoding="utf-8", newline="") as f:
    common_exchanges.to_csv(f)
    f.write("\n" * 3)
    f.write("non-common exchanges\n")
    noncommon_exchanges.to_csv(f)

print(f"wrote {table_path.resolve()}")
common_exchanges

6 common exchanges (present at all 14 locations)
9 non-common exchanges (present at only some locations)
wrote C:\Users\rusai1\PycharmProjects\trailrunner\database\offshore_pipeline_exchanges.csv


location,AZ,DZ,GB,ID,IR,IT,LY,MY,NL,NO,QA,RU,UA,US
exchange_name,,,,,,,,,,,,,,
"Disposal, used mineral oil, 10% water, to hazardous waste incineration",1.16E-6,1.16E-6,1.16E-6,1.16E-6,1.16E-6,1.16E-6,1.16E-6,1.16E-6,1.16E-6,1.16E-6,1.16E-6,1.16E-6,1.16E-6,1.16E-6
"Natural gas, at production",0.0027755,0.0027755,0.0002585,0.0027755,0.0027755,0.0002585,0.0027755,0.0027755,0.0002585,0.0002585,0.0027755,0.0027755,0.0002585,0.0002585
"Natural gas, burned in gas turbine",0.795,0.795,0.32733,0.795,0.795,0.32733,0.795,0.795,0.32733,0.32733,0.795,0.795,0.32733,0.32733
"Pipeline, natural gas, long distance, high capacity, offshore",1.78E-9,1.78E-9,1.78E-9,1.78E-9,1.78E-9,1.78E-9,1.78E-9,1.78E-9,1.78E-9,1.78E-9,1.78E-9,1.78E-9,1.78E-9,1.78E-9
"Transport, freight, lorry, 16t-32t gross weight, fleet average",1.16E-7,1.16E-7,1.16E-7,1.16E-7,1.16E-7,1.16E-7,1.16E-7,1.16E-7,1.16E-7,1.16E-7,1.16E-7,1.16E-7,1.16E-7,1.16E-7
"Transport, natural gas, offshore pipeline, long distance",1,1,1,1,1,1,1,1,1,1,1,1,1,1


In [14]:
noncommon_exchanges

location,AZ,DZ,GB,ID,IR,IT,LY,MY,NL,NO,QA,RU,UA,US
exchange_name,,,,,,,,,,,,,,
Butane,NaN,1.763E-5,1.642E-6,1.763E-5,1.763E-5,NaN,1.763E-5,1.763E-5,1.642E-6,1.642E-6,1.763E-5,1.763E-5,NaN,1.642E-6
"Carbon dioxide, fossil",NaN,6.3642E-5,5.9274E-6,6.3642E-5,6.3642E-5,NaN,6.3642E-5,6.3642E-5,5.9274E-6,5.9274E-6,6.3642E-5,6.3642E-5,NaN,5.9274E-6
Ethane,NaN,0.00015239,1.4193E-5,0.00015239,0.00015239,NaN,0.00015239,0.00015239,1.4193E-5,1.4193E-5,0.00015239,0.00015239,NaN,1.4193E-5
Mercury,NaN,2.7755E-11,2.585E-12,2.7755E-11,2.7755E-11,NaN,2.7755E-11,2.7755E-11,2.585E-12,2.585E-12,2.7755E-11,2.7755E-11,NaN,2.585E-12
"Methane, bromochlorodifluoro-, Halon 1211",NaN,2.24E-9,2.24E-9,2.24E-9,2.24E-9,NaN,2.24E-9,2.24E-9,2.24E-9,2.24E-9,2.24E-9,2.24E-9,NaN,2.24E-9
"Methane, fossil",NaN,0.0018398,0.00017135,0.0018398,0.0018398,NaN,0.0018398,0.0018398,0.00017135,0.00017135,0.0018398,0.0018398,NaN,0.00017135
"Methane, trifluoro-, HFC-23",NaN,8.946E-8,8.946E-8,8.946E-8,8.946E-8,NaN,8.946E-8,8.946E-8,8.946E-8,8.946E-8,8.946E-8,8.946E-8,NaN,8.946E-8
"NMVOC, non-methane volatile organic compounds, unspecified origin",NaN,1.2695E-6,1.1823E-7,1.2695E-6,1.2695E-6,NaN,1.2695E-6,1.2695E-6,1.1823E-7,1.1823E-7,1.2695E-6,1.2695E-6,NaN,1.1823E-7
Propane,NaN,3.4284E-5,3.1932E-6,3.4284E-5,3.4284E-5,NaN,3.4284E-5,3.4284E-5,3.1932E-6,3.1932E-6,3.4284E-5,3.4284E-5,NaN,3.1932E-6


## Full corpus, parsed

The schema was verified consistent across a random sample before this parser was written (all 5 `processInformation` subsections present, same attributes on every file), so the run above went straight to the full 11,947 files rather than stopping at a sample. Result persisted to `processInformation.parquet` alongside this notebook (gitignored, regenerate by re-running this notebook).